In [268]:
import logging
import json
import numpy as np
import inspect


from pathlib import Path
from typing import Optional, Dict, Any
from utils.paths import get_paths
from utils.file_io import save_data
from utils.logging_setup import configure_logging, log_layer_paths
from utils.pipeline_config_loader import (
    load_pipeline_config,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.truths import (
    make_process_run_id,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record_by_hash,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.synthetic_profiles import (
    load_rich_profile_csv,
    load_and_merge_rich_profiles,
    load_correlation_pairs_csv,
    load_group_map_csv,
    load_fault_pairings_csv,
)

from utils.synthetic_missingness import build_missingness_spec_from_truth_payload

from utils.synthetic_generator import SyntheticGenerator, EpisodeSpec



from utils.synthetic_postgres_writer import (
    ensure_sequence,
    reserve_next_batch_id,
    reserve_cycle_range,
    write_stream_batch,
)

from utils.synthetic_export import export_synthetic_batch_to_parquet
from utils.pipeline_config_loader import build_truth_config_block

from utils.postgres_util import get_engine_from_env, build_postgres_url, execute_sql, read_sql_dataframe
from utils.layer_postgres_writer import write_layer_dataframe, prepare_layer_dataframe



In [269]:
# --- Notebook params ---
STAGE = "synthetic"
DATASET = "pump"
MODE = "train"
PROFILE = "default"

In [270]:
paths = get_paths()

config_obj = load_pipeline_config(
    config_root=paths.configs,
    stage=STAGE,
    dataset=DATASET,
    mode=MODE,
    profile=PROFILE,
    project_root=paths.root,
)
CONFIG = config_obj.data

SYN_CFG = CONFIG["synthetic"]
PATHS = CONFIG["resolved_paths"]
PIPELINE = CONFIG.get("pipeline", {"execution_mode": "batch", "orchestration_mode": "notebook"})

PIPELINE_MODE = PIPELINE["execution_mode"]
DATASET_NAME = str(CONFIG["dataset"]["name"]).strip().lower()
TRUTH_VERSION = CONFIG["versions"]["truth"]

TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])
LOGS_PATH = Path(PATHS["logs_root"])
ARTIFACTS_ROOT = Path(PATHS["artifacts_root"])


TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)



set_wandb_dir_from_config(CONFIG)

print("DATASET_NAME:", DATASET_NAME)
print("TRUTHS_PATH:", TRUTHS_PATH)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)

DATASET_NAME: pump
TRUTHS_PATH: /workspace/artifacts/truths
ARTIFACTS_ROOT: /workspace/artifacts


In [271]:
# Logging Setup

# Create gold log path 
synthetic_log_path = paths.logs / "synthetic_data_generator.log"

# Initial Logger
configure_logging(
    "capstone",
    synthetic_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.synthetic")

# Log load and initiation
logger.info("Synethetic Data Generation starting")

# Log paths loads
log_layer_paths(paths, current_layer="synthetic", logger=logger)


2026-03-18 05:48:14,958 | INFO | capstone.synthetic | Synethetic Data Generation starting
2026-03-18 05:48:14,960 | INFO | capstone.synthetic | Project Root Path Loaded: /workspace
2026-03-18 05:48:14,962 | INFO | capstone.synthetic | Project Logging Path Loaded: /workspace/logs
2026-03-18 05:48:14,965 | INFO | capstone.synthetic | Project Artifacts Path Loaded: /workspace/artifacts
2026-03-18 05:48:14,966 | INFO | capstone.synthetic | Project Notebooks Path Loaded: /workspace/notebooks
2026-03-18 05:48:14,968 | INFO | capstone.synthetic | Project Truths Path Loaded: /workspace/artifacts/truths
2026-03-18 05:48:14,969 | INFO | capstone.synthetic | Project Data Path Loaded: /workspace/data
2026-03-18 05:48:14,970 | INFO | capstone.synthetic | Previous Layer (Gold) Path Loaded: /workspace/data/gold


In [272]:
def sample_int_range(rng: np.random.Generator, value_or_range, *, low_inclusive: bool = True) -> int:
    """
    Accepts either:
      - int
      - [low, high]
    Returns an int.
    """
    if isinstance(value_or_range, (int, np.integer)):
        return int(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = int(value_or_range[0])
        high = int(value_or_range[1])
        if low_inclusive:
            # numpy integers are low inclusive, high exclusive
            return int(rng.integers(low, high + 1))
        return int(rng.integers(low, high))

    raise TypeError(f"Expected int or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")


def sample_float_range(rng: np.random.Generator, value_or_range) -> float:
    """
    Accepts either:
      - float/int
      - [low, high]
    Returns a float.
    """
    if isinstance(value_or_range, (float, int, np.floating, np.integer)):
        return float(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = float(value_or_range[0])
        high = float(value_or_range[1])
        return float(rng.uniform(low, high))

    raise TypeError(f"Expected float or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")

In [273]:
# Get Latest Truth Hash

def get_latest_truth_hash(
    *,
    truth_index_path: Path,
    layer_name: str,
    dataset_name: str,
) -> str:
    """
    Return the most recent truth_hash for a given layer + dataset from truth_index.jsonl.

    Assumes truth_index.jsonl is append-only and newer entries are later in the file.
    """
    if not truth_index_path.exists():
        raise FileNotFoundError(f"truth_index.jsonl not found: {truth_index_path}")

    dataset_name_norm = str(dataset_name).strip().lower()
    layer_name_norm = str(layer_name).strip().lower()

    latest_record: Optional[Dict[str, Any]] = None

    with truth_index_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            rec_layer = str(rec.get("layer_name", "")).strip().lower()
            rec_dataset = str(rec.get("dataset_name", "")).strip().lower()

            if rec_layer == layer_name_norm and rec_dataset == dataset_name_norm:
                latest_record = rec

    if latest_record is None:
        raise ValueError(
            f"No truth records found for layer='{layer_name}' dataset='{dataset_name}' in {truth_index_path}"
        )

    truth_hash = latest_record.get("truth_hash")
    if truth_hash is None or str(truth_hash).strip() == "":
        raise ValueError("Latest record is missing a usable truth_hash.")

    return str(truth_hash).strip()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


def get_latest_silver_eda_truth_hash(
    *,
    truth_index_path: Path,
    dataset_name: str,
) -> str:
    """
    Convenience wrapper: latest Silver EDA truth hash for a dataset.
    """
    return get_latest_truth_hash(
        truth_index_path=truth_index_path,
        layer_name="silver_eda",
        dataset_name=dataset_name,
    )

In [274]:
# Updated
# --- Notebook params ---
STAGE = SYN_CFG["layer_name"]
DATASET = "pump"
MODE = "train"
PROFILE = "default"

# Parent truth hash from your latest Silver EDA run
SILVER_EDA_TRUTH_HASH = get_latest_silver_eda_truth_hash(truth_index_path=TRUTH_INDEX_PATH, dataset_name=DATASET_NAME,)
print("Latest SILVER_EDA_TRUTH_HASH:", SILVER_EDA_TRUTH_HASH)

# Faults
# Episode overrides (easy test knobs)
PRIMARY_SENSOR = None          # None => first sensor
PRIMARY_FAULT_TYPE = list(SYN_CFG["faults"]["allowed"])


# Episode Settings
NORMAL_BEFORE = list(SYN_CFG["episode"]["normal_before_range"])
BUILDUP = list(SYN_CFG["episode"]["buildup_range"])
FAILURE = list(SYN_CFG["episode"]["failure_range"])
RECOVERY = list(SYN_CFG["episode"]["recovery_range"])
NORMAL_AFTER = list(SYN_CFG["episode"]["normal_after_range"])
MAGNITUDE = list(SYN_CFG["episode"]["magnitude_range"])

SYNTH_PROCESS_RUN_ID = make_process_run_id(SYN_CFG["process_run_id_prefix"])

# Outputs
OUTPUT_MODE = SYN_CFG["output_mode"]

# Postgres settings
PG_SCHEMA = str(SYN_CFG["postgres"]["schema"])
TABLE_ARTIFACT_NAME = str(SYN_CFG["postgres"]["table_artifact_name"])

# medallion naming: synthetic_<dataset>_<artifact_name>
ARTIFACT_NAME = "stream"       

# Export
EXPORT_ENABLED = bool(SYN_CFG["export"]["enabled"])
EXPORT_DIRECTORY = str(SYN_CFG["export"]["export_dir_key"])

Latest SILVER_EDA_TRUTH_HASH: df2f416c588a68ab07bf03afa75c23587ffb7609aa2eaca7edf0c5ed018d3188


In [275]:
if SILVER_EDA_TRUTH_HASH is None or str(SILVER_EDA_TRUTH_HASH).strip() == "":
    raise ValueError("Set SILVER_EDA_TRUTH_HASH in the parameter cell.")

silver_eda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver_eda",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_EDA_TRUTH_HASH).strip(),
)


PARENT_TRUTH_HASH = get_truth_hash(silver_eda_truth)

SILVER_PREEDA_TRUTH_HASH = get_parent_truth_hash(silver_eda_truth)

silver_preeda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_PREEDA_TRUTH_HASH).strip(),
)

missingness_payload = (silver_preeda_truth.get("runtime_facts", {}) or {}).get("missingness_quarantine", {}) or {}
missingness_spec = build_missingness_spec_from_truth_payload(missingness_payload)


parent_mode = get_pipeline_mode_from_truth(silver_eda_truth)
if parent_mode:
    PIPELINE_MODE = parent_mode

print("PARENT_TRUTH_HASH:", PARENT_TRUTH_HASH)
print("PIPELINE_MODE:", PIPELINE_MODE)

logger.info("W&B PARENT_TRUTH_HASH: %s", PARENT_TRUTH_HASH)
logger.info("W&B PIPELINE_MODE: %s", PIPELINE_MODE)

2026-03-18 05:48:15,102 | INFO | capstone.synthetic | W&B PARENT_TRUTH_HASH: df2f416c588a68ab07bf03afa75c23587ffb7609aa2eaca7edf0c5ed018d3188
2026-03-18 05:48:15,105 | INFO | capstone.synthetic | W&B PIPELINE_MODE: batch


PARENT_TRUTH_HASH: df2f416c588a68ab07bf03afa75c23587ffb7609aa2eaca7edf0c5ed018d3188
PIPELINE_MODE: batch


In [276]:
keys = SYN_CFG["silver_eda_artifact_keys"]

profile_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_normal"])
profile_abnormal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_abnormal"])
profile_recovery_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_recovery"])

corr_pairs_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["corr_pairs_normal"])
group_map_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["group_map_normal"])
fault_pairings_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["fault_pairings_normal"])

print(profile_normal_path)
print(profile_abnormal_path)
print(profile_recovery_path)
print(corr_pairs_normal_path)
print(group_map_normal_path)
print(fault_pairings_normal_path)

logger.info("silver_eda_artifact_keys: %s", keys)


2026-03-18 05:48:15,127 | INFO | capstone.synthetic | silver_eda_artifact_keys: {'profile_normal': 'feature_profile_normal_path', 'profile_abnormal': 'feature_profile_abnormal_path', 'profile_recovery': 'feature_profile_recovery_path', 'corr_pairs_normal': 'sensor_correlation_pairs_normal_path', 'group_map_normal': 'sensor_group_map_normal_path', 'fault_pairings_normal': 'sensor_fault_pairings_normal_path'}


/workspace/artifacts/silver_eda/pump/feature_profile_normal.csv
/workspace/artifacts/silver_eda/pump/feature_profile_abnormal.csv
/workspace/artifacts/silver_eda/pump/feature_profile_recovery.csv
/workspace/artifacts/silver_eda/pump/sensor_correlation_pairs_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_group_map_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_fault_pairings_normal.csv


In [277]:
profile_dir = Path(profile_normal_path).parent

dropped_profile_normal = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__normal.csv"
dropped_profile_abnormal = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__abnormal.csv"
dropped_profile_recovery = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__recovery.csv"

dropped_profile_normal_path = str(dropped_profile_normal) if dropped_profile_normal.exists() else None
dropped_profile_abnormal_path = str(dropped_profile_abnormal) if dropped_profile_abnormal.exists() else None
dropped_profile_recovery_path = str(dropped_profile_recovery) if dropped_profile_recovery.exists() else None

normal_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_normal_path),
    state_scope="normal",
    dropped_profile_csv_path=dropped_profile_normal_path,
)

abnormal_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_abnormal_path),
    state_scope="abnormal",
    dropped_profile_csv_path=dropped_profile_abnormal_path,
)

recovery_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_recovery_path),
    state_scope="recovery",
    dropped_profile_csv_path=dropped_profile_recovery_path,
)

print("Dropped profile files found:",
      dropped_profile_normal.exists(),
      dropped_profile_abnormal.exists(),
      dropped_profile_recovery.exists())


corr_pairs_df = load_correlation_pairs_csv(corr_pairs_normal_path)
group_map_df = load_group_map_csv(group_map_normal_path)
fault_pairings_df = load_fault_pairings_csv(fault_pairings_normal_path)

generator = SyntheticGenerator(
    normal_profiles=normal_profiles,
    abnormal_profiles=abnormal_profiles,
    recovery_profiles=recovery_profiles,
    correlation_pairs_dataframe=corr_pairs_df,
    group_map_dataframe=group_map_df,
    fault_pairings_dataframe=fault_pairings_df,
    random_seed=int(SYN_CFG["random_seed"]),
)

print("Sensors:", len(generator.sensors))
print("First sensors:", generator.sensors[:10])

logger.info("Generator Run")
logger.info("Generator Sensors Count: %s", len(generator.sensors))
logger.info("Generator Sensors List: %s", generator.sensors)

Dropped profile files found: True True True


2026-03-18 05:48:15,548 | INFO | capstone.synthetic | Generator Run
2026-03-18 05:48:15,550 | INFO | capstone.synthetic | Generator Sensors Count: 52
2026-03-18 05:48:15,551 | INFO | capstone.synthetic | Generator Sensors List: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51']


Sensors: 52
First sensors: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09']


In [278]:


print(inspect.signature(SyntheticGenerator.__init__))

logger.info("Generator Signature Inspection: %s", inspect.signature(SyntheticGenerator.__init__))

2026-03-18 05:48:15,567 | INFO | capstone.synthetic | Generator Signature Inspection: (self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', random_seed: 'int' = 42, missingness_spec: 'Optional[MissingnessSpec]' = None) -> 'None'


(self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', random_seed: 'int' = 42, missingness_spec: 'Optional[MissingnessSpec]' = None) -> 'None'


In [279]:
rng = np.random.default_rng(int(SYN_CFG.get("random_seed", 42)))

ep_cfg = SYN_CFG["episode"]

NORMAL_BEFORE = sample_int_range(rng, ep_cfg["normal_before_range"])
BUILDUP       = sample_int_range(rng, ep_cfg["buildup_range"])
FAILURE       = sample_int_range(rng, ep_cfg["failure_range"])
RECOVERY      = sample_int_range(rng, ep_cfg["recovery_range"])
NORMAL_AFTER  = sample_int_range(rng, ep_cfg["normal_after_range"])
MAGNITUDE     = sample_float_range(rng, ep_cfg["magnitude_range"])

sample_ranges_dict = {
    "normal_before_range": ep_cfg["normal_before_range"],
    "normal_before_selection": NORMAL_BEFORE,
    "buildup_range": ep_cfg["buildup_range"],
    "buildup_selection": BUILDUP,
    "failure_range": ep_cfg["failure_range"],
    "failure_selection": FAILURE,
    "recovery_range": ep_cfg["recovery_range"],
    "recovery_selection": RECOVERY,
    "normal_after_range": ep_cfg["normal_after_range"],
    "normal_after_selection": NORMAL_AFTER,
    "magnitude_range": ep_cfg["magnitude_range"],
    "magnitude_selection": MAGNITUDE
}

logger.info("Samples Selected From Episode Ranges: %s", sample_ranges_dict)



primary_sensor = PRIMARY_SENSOR or generator.sensors[0]

logger.info("Primary Sensor: %s", primary_sensor)



rng = np.random.default_rng(int(SYN_CFG.get("random_seed", 42)))

allowed_faults = SYN_CFG["faults"]["allowed"]

if PRIMARY_FAULT_TYPE is None or str(PRIMARY_FAULT_TYPE).strip() == "":
    primary_fault_type = str(rng.choice(allowed_faults))
else:
    # If someone accidentally passed a list, fix it here too
    if isinstance(PRIMARY_FAULT_TYPE, (list, tuple)):
        primary_fault_type = str(rng.choice(PRIMARY_FAULT_TYPE))
    else:
        primary_fault_type = str(PRIMARY_FAULT_TYPE)


logger.info("Allowed Fault Types, %s", allowed_faults)
logger.info("Primary Fault Selected, %s", primary_fault_type)


episode = EpisodeSpec(
    primary_sensor=primary_sensor,
    primary_fault_type=primary_fault_type,
    magnitude=MAGNITUDE,
    normal_before=NORMAL_BEFORE,
    buildup=BUILDUP,
    failure=FAILURE,
    recovery=RECOVERY,
    normal_after=NORMAL_AFTER,
)

synthetic_df = generator.generate_episode(episode)

print("Generated rows:", len(synthetic_df))
display(synthetic_df.head(10))
display(synthetic_df["stream_state"].value_counts())

logger.info("Episodes Generatored : %s", sample_ranges_dict)
logger.info("Generatored Rows: %s", len(synthetic_df))
logger.info("Stream State Value Counts : %s", synthetic_df["stream_state"].value_counts())


2026-03-18 05:48:15,590 | INFO | capstone.synthetic | Samples Selected From Episode Ranges: {'normal_before_range': [200, 600], 'normal_before_selection': 235, 'buildup_range': [100, 240], 'buildup_selection': 209, 'failure_range': [1, 3], 'failure_selection': 2, 'recovery_range': [150, 300], 'recovery_selection': 216, 'normal_after_range': [200, 600], 'normal_after_selection': 373, 'magnitude_range': [0.75, 2.0], 'magnitude_selection': 1.6217100363242047}
2026-03-18 05:48:15,594 | INFO | capstone.synthetic | Primary Sensor: sensor_00
2026-03-18 05:48:15,597 | INFO | capstone.synthetic | Allowed Fault Types, ['drift_up', 'drift_down', 'spike', 'stuck_constant', 'variance_burst', 'step_shift', 'intermittent_dropout', 'sawtooth']
2026-03-18 05:48:15,598 | INFO | capstone.synthetic | Primary Fault Selected, drift_up


Generated rows: 1035


/workspace/utils/synthetic_generator.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[sensor] = x


,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,...,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,stream_state,phase
0,2.372556,48.263734,51.506504,44.185969,614.584808,75.559461,13.339299,15.964418,15.293820,14.880604,...,40.035559,40.514067,46.360219,43.989731,141.470275,53.940796,236.733090,191.765735,normal,normal
1,2.393907,48.280326,51.382927,44.098463,620.944733,75.721603,13.436110,16.015334,15.348859,14.933754,...,40.095059,41.214166,47.013101,43.926545,142.024423,53.930064,230.296911,192.911540,normal,normal
2,2.389921,48.268489,51.626592,44.333097,619.820970,75.808615,13.426407,16.003570,15.333303,14.918762,...,40.252534,40.082909,45.527791,43.853833,143.824247,53.775362,225.022573,193.926341,normal,normal
3,2.401749,48.281336,51.461590,44.162725,621.729043,75.662035,13.466113,16.032557,15.357036,14.946347,...,40.327878,42.047978,47.772697,43.815371,144.573399,53.872290,211.329266,194.971850,normal,normal
4,2.378225,48.290100,51.571500,44.242129,618.101343,75.613072,13.402426,15.980724,15.312455,14.889841,...,40.349550,42.163462,47.790289,43.811508,143.979530,53.897900,206.464174,196.150536,normal,normal
5,2.419076,48.270127,51.385536,44.091390,626.280993,75.571235,13.554323,16.082319,15.419915,14.996531,...,40.402367,40.799922,46.035560,43.660083,143.691892,54.022544,211.616914,196.022874,normal,normal
6,2.437784,48.285554,51.522229,44.221667,630.069501,75.577132,13.618847,16.124802,15.457913,15.031749,...,40.289464,42.033966,47.608527,43.459656,145.002265,53.941973,233.621765,197.268087,normal,normal
7,2.419512,48.294041,51.729585,44.417283,627.020165,75.583518,13.550428,16.076069,15.417020,14.984807,...,40.413760,40.563947,45.590168,43.362237,145.222484,53.989197,227.135948,197.320515,normal,normal
8,2.388136,48.291231,51.603207,44.331950,620.188686,75.804014,13.443969,15.997955,15.336239,14.910052,...,40.283111,40.598897,45.380905,43.221924,145.195038,53.889618,228.823335,197.666684,normal,normal
9,2.390373,48.274322,51.453780,44.200424,620.143055,75.704815,13.451653,16.007822,15.346379,14.929946,...,40.301161,40.728812,45.299544,43.131948,144.874321,53.908488,233.984292,197.759495,normal,normal


stream_state
normal      608
recovery    216
buildup     209
abnormal      2
Name: count, dtype: int64

2026-03-18 05:48:15,757 | INFO | capstone.synthetic | Episodes Generatored : {'normal_before_range': [200, 600], 'normal_before_selection': 235, 'buildup_range': [100, 240], 'buildup_selection': 209, 'failure_range': [1, 3], 'failure_selection': 2, 'recovery_range': [150, 300], 'recovery_selection': 216, 'normal_after_range': [200, 600], 'normal_after_selection': 373, 'magnitude_range': [0.75, 2.0], 'magnitude_selection': 1.6217100363242047}
2026-03-18 05:48:15,759 | INFO | capstone.synthetic | Generatored Rows: 1035
2026-03-18 05:48:15,762 | INFO | capstone.synthetic | Stream State Value Counts : stream_state
normal      608
recovery    216
buildup     209
abnormal      2
Name: count, dtype: int64


In [280]:
# --- helpers required by choose_batch_id ---
from utils.postgres_util import execute_sql, read_sql_dataframe

def _qual(schema: str, name: str) -> str:
    return f'"{schema}"."{name}"'

def table_exists(engine, *, schema: str, table_name: str) -> bool:
    sql = """
    SELECT EXISTS(
      SELECT 1
      FROM information_schema.tables
      WHERE table_schema = :schema AND table_name = :table
    ) AS exists_flag
    """
    df = read_sql_dataframe(engine, sql, params={"schema": schema, "table": table_name})
    return bool(df.loc[0, "exists_flag"])

def drop_table(engine, *, schema: str, table_name: str) -> None:
    execute_sql(engine, f'DROP TABLE IF EXISTS "{schema}"."{table_name}" CASCADE')

def get_max_batch_id(engine, *, schema: str, table_name: str, batch_col: str = "batch_id") -> int:
    fq = _qual(schema, table_name)
    df = read_sql_dataframe(engine, f"SELECT COALESCE(MAX({batch_col}), 0) AS max_batch_id FROM {fq}")
    return int(df.loc[0, "max_batch_id"])


# --- your keepers ---
def canonicalize_existing_batch_ids(engine, *, schema: str, table_name: str, batch_col: str = "batch_id") -> Dict[str, int]:
    fq = f'"{schema}"."{table_name}"'
    sql = f"""
    WITH b AS (
        SELECT DISTINCT {batch_col} AS old_batch_id
        FROM {fq}
        WHERE {batch_col} IS NOT NULL
    ),
    m AS (
        SELECT old_batch_id,
               DENSE_RANK() OVER (ORDER BY old_batch_id ASC) AS new_batch_id
        FROM b
    )
    UPDATE {fq} t
    SET {batch_col} = m.new_batch_id
    FROM m
    WHERE t.{batch_col} = m.old_batch_id
    """
    execute_sql(engine, sql)

    df = read_sql_dataframe(
        engine,
        f"""
        SELECT
          COUNT(DISTINCT {batch_col}) AS distinct_batches,
          COALESCE(MIN({batch_col}), 0) AS min_batch_id,
          COALESCE(MAX({batch_col}), 0) AS max_batch_id
        FROM {fq}
        """
    )
    return {
        "distinct_batches": int(df.loc[0, "distinct_batches"]),
        "min_batch_id": int(df.loc[0, "min_batch_id"]),
        "max_batch_id": int(df.loc[0, "max_batch_id"]),
    }


def choose_batch_id(
    engine,
    *,
    schema: str,
    table_name: str,
    write_mode: str,   # "reset" | "append"
    append_mode: str,  # "continue" | "renumber" (only used when write_mode="append")
    batch_col: str = "batch_id",
) -> int:
    write_mode = str(write_mode).strip().lower()
    append_mode = str(append_mode).strip().lower()

    if write_mode == "reset":
        drop_table(engine, schema=schema, table_name=table_name)
        return 1

    if write_mode != "append":
        raise ValueError("write_mode must be 'reset' or 'append'")

    if not table_exists(engine, schema=schema, table_name=table_name):
        return 1

    if append_mode == "continue":
        return get_max_batch_id(engine, schema=schema, table_name=table_name, batch_col=batch_col) + 1

    if append_mode == "renumber":
        summary = canonicalize_existing_batch_ids(engine, schema=schema, table_name=table_name, batch_col=batch_col)
        return int(summary["max_batch_id"]) + 1

    raise ValueError("append_mode must be 'continue' or 'renumber'")

In [281]:
engine = get_engine_from_env()

PG_SCHEMA = "capstone"
ARTIFACT_NAME = "stream"
TABLE_NAME = f"synthetic_{DATASET_NAME.lower()}_{ARTIFACT_NAME}"

# ---- Your flags ----
WRITE_MODE = "append"       # "reset" | "append"
APPEND_MODE = "renumber"    # "continue" | "renumber"   (only matters if WRITE_MODE="append")

batch_id = choose_batch_id(
    engine,
    schema=PG_SCHEMA,
    table_name=TABLE_NAME,
    write_mode=WRITE_MODE,
    append_mode=APPEND_MODE,
    batch_col="batch_id",
)

print("Chosen batch_id:", batch_id)

# cycles can stay sequence-based for now
ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id")
cycle_start = reserve_cycle_range(
    engine,
    schema=PG_SCHEMA,
    sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id",
    n_rows=len(synthetic_df),
)

table_written = write_stream_batch(
    engine,
    synthetic_df,
    dataset_name=DATASET_NAME,
    schema=PG_SCHEMA,
    artifact_name=ARTIFACT_NAME,
    batch_id=batch_id,
    cycle_start=cycle_start,
)

print("Wrote:", table_written, "batch_id:", batch_id, "cycle_start:", cycle_start)

Chosen batch_id: 3


Wrote: synthetic_pump_stream batch_id: 3 cycle_start: 9316


In [282]:
import os
{k: os.environ.get(k) for k in ["DB_HOST","DB_PORT","DB_NAME","DB_USER","DB_PASSWORD","POSTGRES_DB","POSTGRES_USER","POSTGRES_PASSWORD"]}

{'DB_HOST': 'dcook_capstone_postgres',
 'DB_PORT': '5432',
 'DB_NAME': 'dcook_capstone_postgres_db',
 'DB_USER': 'dcook_admin',
 'DB_PASSWORD': 'V9m!pQ4z@H2eS7wK',
 'POSTGRES_DB': None,
 'POSTGRES_USER': None,
 'POSTGRES_PASSWORD': None}

In [283]:
import importlib.util
print("psycopg:", importlib.util.find_spec("psycopg"))
print("psycopg2:", importlib.util.find_spec("psycopg2"))

psycopg: None
psycopg2: ModuleSpec(name='psycopg2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x783617a5e4d0>, origin='/opt/miniconda/envs/capstone/lib/python3.10/site-packages/psycopg2/__init__.py', submodule_search_locations=['/opt/miniconda/envs/capstone/lib/python3.10/site-packages/psycopg2'])


In [284]:
process_run_id = make_process_run_id(str(SYN_CFG.get("process_run_id_prefix", "synthetic")))

synthetic_truth = initialize_layer_truth(
    truth_version=str(TRUTH_VERSION),
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
    process_run_id=process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=PARENT_TRUTH_HASH,
)

LAYER_NAME = "synthetic"

resolved_config_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
resolved_config_dir.mkdir(parents=True, exist_ok=True)

resolved_config_path = resolved_config_dir / f"{DATASET_NAME}__{LAYER_NAME}__resolved_config.yaml"

# export_config_snapshot requires a destination path
export_config_snapshot(CONFIG, destination=resolved_config_path)

print("Resolved config written to:", resolved_config_path)


synthetic_truth = update_truth_section(
    synthetic_truth,
    "config_snapshot",
    {
        # store the path you just wrote
        "resolved_config_path": str(resolved_config_path),

        # optional: small inline config view (JSON-safe)
        "truth_config_block": build_truth_config_block(CONFIG),

        # keep your synthetic config subset if you want
        "synthetic_cfg": SYN_CFG,
    },
)

synthetic_truth = update_truth_section(
    synthetic_truth,
    "runtime_facts",
    {
        "primary_sensor": primary_sensor,
        "primary_fault_type": primary_fault_type,
        "episode": episode.__dict__,
        "normal_before_range": ep_cfg["normal_before_range"],
        "normal_before_selection": NORMAL_BEFORE,
        "buildup_range": ep_cfg["buildup_range"],
        "buildup_selection": BUILDUP,
        "failure_range": ep_cfg["failure_range"],
        "failure_selection": FAILURE,
        "recovery_range": ep_cfg["recovery_range"],
        "recovery_selection": RECOVERY,
        "normal_after_range": ep_cfg["normal_after_range"],
        "normal_after_selection": NORMAL_AFTER,
        "magnitude_range": ep_cfg["magnitude_range"],
        "magnitude_selection": MAGNITUDE,
        "row_count": int(len(synthetic_df)),
        "parent_truth_hash": PARENT_TRUTH_HASH,
        "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    },
)

# Optionally save local parquet artifact too (useful for debugging)
synth_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
synth_dir.mkdir(parents=True, exist_ok=True)
out_path = synth_dir / f"{DATASET_NAME}__synthetic__episode.parquet"
save_data(synthetic_df, synth_dir, out_path.name)

artifact_paths_payload = {
    "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    "profile_normal_path": profile_normal_path,
    "profile_abnormal_path": profile_abnormal_path,
    "profile_recovery_path": profile_recovery_path,
    "corr_pairs_normal_path": corr_pairs_normal_path,
    "group_map_normal_path": group_map_normal_path,
    "fault_pairings_normal_path": fault_pairings_normal_path,
    "synthetic_episode_path": str(out_path),
    "postgres_schema": PG_SCHEMA,
    "postgres_table": table_written,
    "postgres_batch_id": int(batch_id),
    "postgres_cycle_start": int(cycle_start),
}

#export_path = Path(PATHS["data_raw_dir"] / "synthetic") 

if EXPORT_ENABLED:
    artifact_paths_payload["export_batch_parquet_path"] = str(out_path)

synthetic_truth = update_truth_section(synthetic_truth, "artifact_paths", artifact_paths_payload)

meta_columns = sorted(["meta__truth_hash", "meta__parent_truth_hash", "meta__pipeline_mode"])
feature_columns = sorted([c for c in synthetic_df.columns if not str(c).startswith("meta__")])

truth_record = build_truth_record(
    truth_base=synthetic_truth,
    row_count=int(len(synthetic_df)),
    column_count=int(synthetic_df.shape[1] + 3),
    meta_columns=meta_columns,
    feature_columns=feature_columns,
)

synth_truth_hash = truth_record["truth_hash"]

# stamp lineage columns into dataframe (optional)
synthetic_df = stamp_truth_columns(
    synthetic_df,
    truth_hash=synth_truth_hash,
    parent_truth_hash=PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

truth_path = save_truth_record(
    truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
)

append_truth_index(truth_record, truth_index_path=TRUTH_INDEX_PATH)

print("Synthetic truth hash:", synth_truth_hash)
print("Synthetic truth path:", truth_path)
print("Local episode parquet:", out_path)

2026-03-18 05:48:18,281 | INFO | capstone.file_io | Saving DataFrame to Parquet: /workspace/artifacts/synthetic/pump/pump__synthetic__episode.parquet


Resolved config written to: /workspace/artifacts/synthetic/pump/pump__synthetic__resolved_config.yaml


2026-03-18 05:48:18,413 | INFO | capstone.file_io | Saved: pump__synthetic__episode.parquet | shape=(1035, 54) | columns=['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51', 'stream_state', 'phase']


Synthetic truth hash: 464366243cd619606817a59ac385b86e9abb8a3c68a6d52ef0a9ae843df5fc39
Synthetic truth path: /workspace/artifacts/truths/synthetic/pump__synthetic__truth__464366243cd619606817a59ac385b86e9abb8a3c68a6d52ef0a9ae843df5fc39.json
Local episode parquet: /workspace/artifacts/synthetic/pump/pump__synthetic__episode.parquet


In [285]:
from utils.synthetic_profiles import load_and_merge_rich_profiles
from utils.synthetic_generator import SyntheticGenerator, EpisodeSpec
from utils.synthetic_missingness import build_missingness_spec_from_truth_payload
from utils.synthetic_export import export_synthetic_batch_to_parquet

print("Imports OK")

Imports OK
